# Vision Transformer для классификации фруктов и овощей

Ноутбук оформлен по структуре задания на пятерку из лабораторной работы 1: выбор начальных условий, baseline, улучшение baseline и собственная имплементация модели.

## 1. Выбор начальных условий

Фиксируем задачу многоклассовой классификации фруктов и овощей, путь к датасету, устройство обучения и основные гиперпараметры transformer-экспериментов. Для `torchvision` ViT используем transfer learning, так как обучение ViT с нуля на малом датасете дает низкое качество.

Подключаем библиотеки и локальные модули проекта, фиксируем параметры эксперимента и загружаем датасет через Kaggle API, если он еще не находится в `data/`.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fruitveg_lab.data import create_dataloaders, describe_dataset, ensure_kaggle_dataset, make_transforms
from fruitveg_lab.models import (
    TinyVisionTransformer,
    configure_vit_finetuning,
    count_trainable_parameters,
    create_torchvision_vit,
)
from fruitveg_lab.plotting import plot_confusion_matrix, plot_history, show_image_grid
from fruitveg_lab.training import RunConfig, fit_classifier, get_device, make_conclusion, print_report, seed_everything, summarize_results

FAST_DEV_RUN = False
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
BASE_EPOCHS = 5
SEED = 42

DATA_DIR = PROJECT_ROOT / "data" / "fruit-and-vegetable-image-recognition"
DATA_ROOT = ensure_kaggle_dataset(DATA_DIR)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "transformer"
DEVICE = get_device()

warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes should be converted to RGBA images",
    category=UserWarning,
    module="PIL.Image",
)

seed_everything(SEED)
print(f"project: {PROJECT_ROOT}")
print(f"dataset: {DATA_ROOT}")
print(f"device: {DEVICE}")

### 1.a. Набор данных и практическая задача

Датасет используется для распознавания продукта по фотографии. Практический смысл: кассы самообслуживания, складской учет, приложения питания и автоматическая маркировка товаров.

Проверяем структуру датасета и формируем таблицу количества изображений по классам в `train`, `validation` и `test`.

In [ ]:
info = describe_dataset(DATA_ROOT)
counts = pd.DataFrame(info.split_counts).fillna(0).astype(int)
counts["total"] = counts.sum(axis=1)
display(counts)
print(f"classes: {len(info.classes)}")
print(f"train images: {counts['train'].sum()}")
print(f"validation images: {counts['validation'].sum()}")
print(f"test images: {counts['test'].sum()}")

### 1.b. Метрики качества

Основные метрики: `accuracy`, macro/weighted `precision`, `recall`, `F1`, `top-3 accuracy`, classification report и confusion matrix. Для выбора лучшей конфигурации используется `validation macro F1`, потому что все классы важны одинаково.

Создаем DataLoader-объекты для выбранного режима преобразований и визуализируем примеры обучающих изображений.

In [ ]:
def loaders_for(transform_mode: str, batch_size: int = BATCH_SIZE):
    train_transform = make_transforms(IMAGE_SIZE, mode=transform_mode, normalize=True)
    eval_transform = make_transforms(IMAGE_SIZE, mode="plain", normalize=True)
    return create_dataloaders(
        DATA_ROOT,
        train_transform=train_transform,
        eval_transform=eval_transform,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
    )

preview_loaders = loaders_for("plain", batch_size=12)
show_image_grid(preview_loaders.train, preview_loaders.class_names, count=12, normalized=True)

## 2. Создание бейзлайна и оценка качества

Пункты 2.a-2.b: обучаем transformer baseline `ViT-B/16` из `torchvision` с ImageNet-весами. На baseline-этапе замораживаем backbone и обучаем только классификационную голову на 36 классов.

Обучаем baseline `torchvision ViT-B/16` в режиме feature extraction: используем ImageNet-веса, замораживаем backbone, обучаем classifier head и оцениваем качество на test split.

In [ ]:
num_classes = len(preview_loaders.class_names)
plain_loaders = loaders_for("plain")

baseline_model = create_torchvision_vit(
    "vit_b_16",
    num_classes=num_classes,
    image_size=IMAGE_SIZE,
    weights="imagenet",
)
baseline_model = configure_vit_finetuning(baseline_model, mode="head")
trainable, total = count_trainable_parameters(baseline_model)
print(f"trainable parameters: {trainable:,} / {total:,}")

vit_baseline = fit_classifier(
    baseline_model,
    name="torchvision_vit_b16_imagenet_head",
    train_loader=plain_loaders.train,
    val_loader=plain_loaders.val,
    test_loader=plain_loaders.test,
    class_names=plain_loaders.class_names,
    config=RunConfig(
        epochs=BASE_EPOCHS,
        learning_rate=1e-3,
        weight_decay=1e-4,
        scheduler="none",
        fast_dev_run=FAST_DEV_RUN,
        seed=SEED,
    ),
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

plot_history(vit_baseline)
print_report(vit_baseline)
plot_confusion_matrix(vit_baseline)

## 3. Улучшение бейзлайна

Пункты 3.a-3.g: формулируем и проверяем гипотезы про аугментации, cosine scheduler, разморозку последних encoder-блоков и `ViT-B/32`; лучшую конфигурацию выбираем по `validation macro F1` и сравниваем с baseline.

Проводим серию transformer-экспериментов с предобученными весами, разными режимами fine-tuning и аугментациями, затем формируем таблицы для validation и test split.

In [ ]:
vit_specs = [
    {
        "name": "vit_b16_imagenet_head_augmented",
        "model": "vit_b_16",
        "weights": "imagenet",
        "tuning_mode": "head",
        "trainable_blocks": 0,
        "transform_mode": "augmented",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "scheduler": "cosine",
        "epochs": BASE_EPOCHS,
    },
    {
        "name": "vit_b16_imagenet_last2",
        "model": "vit_b_16",
        "weights": "imagenet",
        "tuning_mode": "last_blocks",
        "trainable_blocks": 2,
        "transform_mode": "plain",
        "learning_rate": 2e-5,
        "weight_decay": 1e-4,
        "scheduler": "cosine",
        "epochs": BASE_EPOCHS + 2,
    },
    {
        "name": "vit_b16_imagenet_last4_augmented",
        "model": "vit_b_16",
        "weights": "imagenet",
        "tuning_mode": "last_blocks",
        "trainable_blocks": 4,
        "transform_mode": "augmented",
        "learning_rate": 1e-5,
        "weight_decay": 2e-4,
        "scheduler": "cosine",
        "epochs": BASE_EPOCHS + 2,
    },
    {
        "name": "vit_b32_imagenet_head_augmented",
        "model": "vit_b_32",
        "weights": "imagenet",
        "tuning_mode": "head",
        "trainable_blocks": 0,
        "transform_mode": "augmented",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "scheduler": "cosine",
        "epochs": BASE_EPOCHS,
    },
]

vit_experiment_records = []
for spec in vit_specs:
    loaders = loaders_for(spec["transform_mode"])
    model = create_torchvision_vit(
        spec["model"],
        num_classes=num_classes,
        image_size=IMAGE_SIZE,
        weights=spec["weights"],
    )
    model = configure_vit_finetuning(
        model,
        mode=spec["tuning_mode"],
        trainable_blocks=spec["trainable_blocks"],
    )
    trainable, total = count_trainable_parameters(model)
    print(f"{spec['name']}: trainable parameters {trainable:,} / {total:,}")

    result = fit_classifier(
        model,
        name=spec["name"],
        train_loader=loaders.train,
        val_loader=loaders.val,
        test_loader=loaders.test,
        class_names=loaders.class_names,
        config=RunConfig(
            epochs=spec["epochs"],
            learning_rate=spec["learning_rate"],
            weight_decay=spec["weight_decay"],
            scheduler=spec["scheduler"],
            fast_dev_run=FAST_DEV_RUN,
            seed=SEED,
        ),
        device=DEVICE,
        output_dir=OUTPUT_DIR,
    )
    vit_experiment_records.append({"spec": spec, "result": result})

vit_hypothesis_results = [record["result"] for record in vit_experiment_records]
display(summarize_results([vit_baseline] + vit_hypothesis_results, split="val"))
display(summarize_results([vit_baseline] + vit_hypothesis_results, split="test"))

### 3.c-3.g. Формирование и оценка улучшенного бейзлайна

После проверки гипотез выбираем лучшую конфигурацию, затем выводим ее подробные метрики и графики.

Выбираем лучший transformer baseline по `validation macro F1`, затем выводим его настройки, отчет классификации, историю обучения и confusion matrix.

In [ ]:
best_record = max(vit_experiment_records, key=lambda item: item["result"]["val"]["macro_f1"])
best_vit_spec = best_record["spec"]
best_torchvision_vit = best_record["result"]

print("Best ViT spec:")
display(pd.DataFrame([best_vit_spec]))
print_report(best_torchvision_vit)
plot_history(best_torchvision_vit)
plot_confusion_matrix(best_torchvision_vit)

## 4. Имплементация алгоритма машинного обучения

Пункты 4.a-4.j: обучаем собственную компактную Vision Transformer с patch embedding, class token, позиционными эмбеддингами, attention-блоками и MLP-head. Собственная модель обучается с нуля, поэтому ожидаемо уступает transfer learning baseline.

Обучаем собственную Vision Transformer в базовом режиме, после чего выводим метрики, графики и confusion matrix.

In [ ]:
custom_vit_plain = fit_classifier(
    TinyVisionTransformer(
        num_classes=num_classes,
        image_size=IMAGE_SIZE,
        patch_size=16,
        embed_dim=192,
        depth=4,
        num_heads=4,
        dropout=0.10,
    ),
    name="custom_tiny_vit_plain",
    train_loader=plain_loaders.train,
    val_loader=plain_loaders.val,
    test_loader=plain_loaders.test,
    class_names=plain_loaders.class_names,
    config=RunConfig(
        epochs=BASE_EPOCHS,
        learning_rate=3e-4,
        weight_decay=1e-4,
        scheduler="none",
        fast_dev_run=FAST_DEV_RUN,
        seed=SEED,
    ),
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

plot_history(custom_vit_plain)
print_report(custom_vit_plain)
plot_confusion_matrix(custom_vit_plain)

Обучаем собственную Vision Transformer с преобразованиями и scheduler, выбранными на этапе улучшения baseline; learning rate оставляем достаточным для обучения модели с нуля.

In [ ]:
improved_loaders = loaders_for(best_vit_spec["transform_mode"])
custom_improved_lr = max(best_vit_spec["learning_rate"], 3e-4)

custom_vit_improved = fit_classifier(
    TinyVisionTransformer(
        num_classes=num_classes,
        image_size=IMAGE_SIZE,
        patch_size=16,
        embed_dim=192,
        depth=4,
        num_heads=4,
        dropout=0.15,
    ),
    name="custom_tiny_vit_improved",
    train_loader=improved_loaders.train,
    val_loader=improved_loaders.val,
    test_loader=improved_loaders.test,
    class_names=improved_loaders.class_names,
    config=RunConfig(
        epochs=best_vit_spec["epochs"],
        learning_rate=custom_improved_lr,
        weight_decay=best_vit_spec["weight_decay"],
        scheduler=best_vit_spec["scheduler"],
        fast_dev_run=FAST_DEV_RUN,
        seed=SEED,
    ),
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

plot_history(custom_vit_improved)
print_report(custom_vit_improved)
plot_confusion_matrix(custom_vit_improved)

### 4.d-4.j. Сравнение и выводы

Сравниваются baseline, улучшенный `torchvision` baseline, собственная модель и собственная модель с улучшениями.

Формируем финальную таблицу результатов transformer-моделей на test split и краткий вывод по значениям метрик.

In [ ]:
vit_final_results = [
    vit_baseline,
    best_torchvision_vit,
    custom_vit_plain,
    custom_vit_improved,
]

display(summarize_results(vit_final_results, split="test"))
print(make_conclusion(vit_final_results, split="test"))